# Phase 0 — Project Setup

**Roadmap reference:** `ContractIQ_Project_Execution_Roadmap.md`, Phase 0

**Goal of this notebook:** verify the environment, load configuration, and produce a validated document manifest for the 36-document dataset. Nothing here parses document content yet — that's Phase 1.

**Completion gate:** manifest accounts for 36 expected documents, secrets are not printed, and a restarted kernel can reproduce the same manifest.

## 1. Make `src/contractiq` importable

This notebook lives in `Research/`, one level below the project root, so `src/` needs to be on the path before we can import the reusable package.

In [1]:
import sys
sys.path.append("..")  # project root, so `src` is importable

from src.contractiq import config
from src.contractiq.ingestion import build_manifest, save_manifest, print_manifest_report

print("Imports OK.")

Imports OK.


## 2. Verify configuration and secrets

Confirms `OPENAI_API_KEY` (and Langfuse keys, if set) loaded correctly from `.env` *without ever printing their values* — required by the submission checklist ("secrets are not printed").

In [2]:
config.assert_secrets_not_exposed()

print(f"DATA_ROOT:        {config.DATA_ROOT}")
print(f"ARTIFACTS_ROOT:   {config.ARTIFACTS_ROOT}")
print(f"CHROMA_PERSIST_DIR: {config.CHROMA_PERSIST_DIR}")
print(f"Expected documents: {config.EXPECTED_TOTAL_DOCUMENTS}")

Secrets loaded from environment; nothing printed. OK to proceed.
DATA_ROOT:        E:\Sheriff_faang\full_end_to_end_project_implementation\IK_PWC_Agentic_AI_Project\PWC_CAPSTONE_PROJECT\IK_PWC_COURSE_CAPSTONE_PROJECTS\datasets\contract_intelligence
ARTIFACTS_ROOT:   E:\Sheriff_faang\full_end_to_end_project_implementation\IK_PWC_Agentic_AI_Project\PWC_CAPSTONE_PROJECT\IK_PWC_COURSE_CAPSTONE_PROJECTS\PWC_CONTRACTIQ_PROJECT\artifacts
CHROMA_PERSIST_DIR: E:\Sheriff_faang\full_end_to_end_project_implementation\IK_PWC_Agentic_AI_Project\PWC_CAPSTONE_PROJECT\IK_PWC_COURSE_CAPSTONE_PROJECTS\PWC_CONTRACTIQ_PROJECT\chroma_contracts_db
Expected documents: 36


## 3. Confirm the eight-category taxonomy

This is the real taxonomy from the problem statement (36 documents, 8 categories) — not invented, sourced from `ContractIQ_Product_Overview.md`'s dataset table. If the official `Contract_Intelligence_Detailed_Problem_Statement.pdf` names these categories differently, update `CATEGORY_TAXONOMY` in `config.py` before continuing.

In [3]:
for spec in config.CATEGORY_TAXONOMY:
    print(f"  {spec.key:<28} {spec.display_name:<28} expected={spec.expected_count}")

  master_level_agreements      Master Level Agreements      expected=6
  transaction_contracts        Transaction Contracts        expected=4
  commercial_documents         Commercial Documents         expected=2
  financial_and_billing        Financial and Billing        expected=4
  operational_documents        Operational Documents        expected=4
  compliance_documents         Compliance Documents         expected=6
  negotiation_history          Negotiation History          expected=6
  governance                   Governance                   expected=4


## 4. Build the document manifest

Scans `DATA_ROOT`, fuzzy-matches subfolder names against the taxonomy above, hashes every file, and assigns a stable `doc_id`. Folders that don't match confidently are reported as `UNSPECIFIED` rather than guessed — fix those in `FOLDER_TO_CATEGORY` (in `config.py`) if they should map to a real category.

In [4]:
documents = build_manifest()
print(f"\nScanned {len(documents)} document(s).")

⚠ Could not confidently match these folder names to the taxonomy — filed under UNSPECIFIED. Add them to FOLDER_TO_CATEGORY in config.py if they should map to a real category:
    'Disputes, audits and governance data'
    'Historical negotiation data'
    'Operational service delivery docs'

Scanned 36 document(s).


## 5. Completion-gate report

Checks the manifest against the expected 36-document, 8-category shape. Any mismatch here must be resolved before Phase 1 — don't parse a dataset you haven't confirmed is complete.

In [5]:
print_manifest_report(documents)

Total documents found: 36 (expected 36)

Per-category counts (found / expected):
  Master Level Agreements       6 / 6
  Transaction Contracts         4 / 4
  Commercial Documents          2 / 2
  Financial and Billing         4 / 4
  Operational Documents         0 / 4  <-- check this
  Compliance Documents          6 / 6
  Negotiation History           0 / 6  <-- check this
  Governance                    0 / 4  <-- check this
  UNSPECIFIED                  14  <-- fix FOLDER_TO_CATEGORY


## 6. Persist the manifest

Saved to `artifacts/manifests/document_manifest.json`. A restarted kernel can reload this instead of re-scanning, and Phase 1 will read from here.

In [ ]:
saved_path = save_manifest(documents)
print(f"Manifest saved to: {saved_path}")

## 7. Spot-check a few records

Quick visual sanity check before moving on — not a substitute for the completion-gate report above, just a human glance at real entries.

In [ ]:
for d in documents[:5]:
    print(d.doc_id, "|", d.category_key, "|", d.document_role.value, "|", d.filename)

---
### Next: `01_Data_Manifest_and_Parsing.ipynb`

Pick one representative MSA and one table-heavy supporting document from this manifest, parse them with `python-docx`, and preserve stable source locations (`paragraph:14`, `table:2,row:4,cell:3`) before running the parser on the full corpus.